# 05 — SHAP Interpretability for Customer Churn

**Business question:** Why is the model predicting that a customer will churn?

This notebook adds model transparency at two levels:
- **Global:** which features drive churn predictions overall?
- **Local:** why does a particular customer receive a high or low churn score?

SHAP explains model predictions; it does **not** establish causal relationships.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from src.features import clean_telco, add_business_features

import shap

## 1. Load customer data

The notebook uses the IBM Telco dataset if available:

`data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv`

Otherwise it falls back to the repository's synthetic demo data.

In [ ]:
ibm_path = ROOT / "data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
demo_path = ROOT / "data/processed/demo_customers.csv"

if ibm_path.exists():
    df = add_business_features(clean_telco(pd.read_csv(ibm_path)))
    print("Using IBM Telco dataset")
else:
    if not demo_path.exists():
        generator = ROOT / "data/generate_demo_data.py"
        ns = {"__file__": str(generator)}
        exec(generator.read_text(), ns)
        ns["main"]()
    df = add_business_features(pd.read_csv(demo_path))
    print("Using synthetic demo dataset")

df.head()

## 2. Prepare modeling features

Customer identifiers are excluded. Numeric features are imputed, while categorical
features are one-hot encoded.

In [ ]:
target = "churn_flag"
drop_cols = [c for c in ["customer_id", "churn"] if c in df.columns]

X = df.drop(columns=drop_cols + [target], errors="ignore")
y = df[target].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

numeric_cols = X_train.select_dtypes(include="number").columns.tolist()
categorical_cols = X_train.select_dtypes(exclude="number").columns.tolist()

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ]), numeric_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]), categorical_cols)
])

## 3. Train a tree-based churn model

XGBoost is preferred for this notebook because TreeSHAP provides efficient explanations
for boosted trees.

In [ ]:
try:
    from xgboost import XGBClassifier

    model = XGBClassifier(
        n_estimators=250,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    )
    model_name = "XGBoost"

except ImportError:
    from sklearn.ensemble import RandomForestClassifier

    model = RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    )
    model_name = "RandomForest"

print("Model:", model_name)

In [ ]:
X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)

model.fit(X_train_t, y_train)
prob = model.predict_proba(X_test_t)[:, 1]

pd.Series({
    "ROC-AUC": roc_auc_score(y_test, prob),
    "PR-AUC": average_precision_score(y_test, prob),
    "Brier Score": brier_score_loss(y_test, prob)
}).round(4)

## 4. Recover readable transformed feature names

In [ ]:
feature_names = (
    pd.Index(preprocessor.get_feature_names_out())
      .str.replace("num__", "", regex=False)
      .str.replace("cat__", "", regex=False)
)

X_train_shap = pd.DataFrame(X_train_t, columns=feature_names, index=X_train.index)
X_test_shap = pd.DataFrame(X_test_t, columns=feature_names, index=X_test.index)

X_test_shap.head()

## 5. Calculate SHAP values

To keep the notebook responsive, explanations are generated for a representative sample
of the test set.

In [ ]:
background = X_train_shap.sample(min(500, len(X_train_shap)), random_state=42)
explain_sample = X_test_shap.sample(min(1000, len(X_test_shap)), random_state=42)

explainer = shap.Explainer(model, background)
shap_values = explainer(explain_sample)

print(shap_values.values.shape)

# Global Interpretability

## 6. Global feature importance

Mean absolute SHAP value measures how strongly each transformed feature affects model
predictions across customers.

In [ ]:
shap.plots.bar(shap_values, max_display=15, show=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "shap_global_importance.png", dpi=200, bbox_inches="tight")
plt.show()

## 7. SHAP beeswarm

The beeswarm shows both the size and direction of feature contributions.

In [ ]:
shap.plots.beeswarm(shap_values, max_display=15, show=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "shap_beeswarm.png", dpi=200, bbox_inches="tight")
plt.show()

## 8. Export global importance table

This output can later be used in the README, executive report, or Power BI.

In [ ]:
importance = pd.DataFrame({
    "feature": explain_sample.columns,
    "mean_abs_shap": np.abs(shap_values.values).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)

importance.to_csv(
    ROOT / "data/processed/shap_global_importance.csv",
    index=False
)

importance.head(20)

# Customer-Level Interpretability

## 9. Select a high-risk customer

In [ ]:
scored = pd.DataFrame({
    "actual_churn": y_test,
    "predicted_churn_probability": prob
}, index=X_test.index)

customer_idx = scored["predicted_churn_probability"].idxmax()
customer_x = X_test_shap.loc[[customer_idx]]
customer_shap = explainer(customer_x)

print("Customer index:", customer_idx)
print("Predicted churn probability:",
      round(scored.loc[customer_idx, "predicted_churn_probability"], 4))
print("Observed churn:", scored.loc[customer_idx, "actual_churn"])

## 10. Waterfall plot

The waterfall plot shows which features push this customer's prediction above or below
the model's baseline.

In [ ]:
shap.plots.waterfall(customer_shap[0], max_display=15, show=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "shap_customer_waterfall.png", dpi=200, bbox_inches="tight")
plt.show()

## 11. Business-readable explanation table

In [ ]:
customer_explanation = pd.DataFrame({
    "feature": customer_x.columns,
    "feature_value": customer_x.iloc[0].values,
    "shap_value": customer_shap.values[0]
})

customer_explanation["absolute_impact"] = customer_explanation["shap_value"].abs()

customer_explanation = (
    customer_explanation
    .sort_values("absolute_impact", ascending=False)
    .drop(columns="absolute_impact")
)

customer_explanation.head(15)

## Interpretation

A practical interpretation should be written in business language, for example:

> The model assigns this customer elevated churn risk. The largest contributors are
> contract structure, tenure, pricing, and service characteristics, while other features
> partially offset that risk.

Do **not** state that the highest-SHAP features *cause* churn. SHAP describes how the
trained model reaches its prediction.

# SHAP vs. Uplift Modeling

These tools answer different questions:

| Business question | Method |
|---|---|
| Who is likely to churn? | Churn classifier |
| Why did the model predict churn? | SHAP |
| Does the retention offer work overall? | Bayesian A/B test |
| Who responds because of the offer? | Uplift / CATE model |
| Who should receive the offer? | Expected-profit optimization |

A feature can strongly predict churn without being a useful intervention target.
This separation is important when moving from prediction to retention strategy.

# Generated Outputs

Running this notebook creates:

```text
reports/figures/shap_global_importance.png
reports/figures/shap_beeswarm.png
reports/figures/shap_customer_waterfall.png
data/processed/shap_global_importance.csv
```

For a portfolio README, the most useful visuals are the global SHAP importance or
beeswarm plot plus one individual waterfall example.